# Final Model Evaluation

This notebook performs the first FINAL MODEL EVALUATION of the preselected fraud detection models on the chronological test split. Portions of the test distribution were inspected earlier during full-dataset EDA and integrity checks, but test data was excluded from model fitting, hyperparameter optimization, early stopping, threshold optimization, model selection, and final metric calculation until this notebook. Results are therefore a held-out evaluation, not a strictly blind holdout.

The evaluation will focus on the models selected during validation:

- Random Forest
- MLP
- LSTM

Average Precision (AP) will remain the primary evaluation metric because of the severe class imbalance in the fraud detection problem.

Precision, Recall, F1-score, ROC-AUC, and confusion matrices will be reported as secondary evaluation metrics.

No model or threshold will be modified based on the final test results.

## 1. Selected Models

The candidate models for final evaluation were selected using validation performance only.

The historical notebook validation reference showed Random Forest leading at Average Precision (AP) 0.858000, followed by MLP at 0.857221 and LSTM at 0.848276. These are historical notebook experiment values; the saved artifacts were generated in separate reproducible runs and have different reproduced validation AP values (Random Forest 0.857966, MLP 0.846886, LSTM 0.846636).

The historical MLP result was therefore retained as a competitive tabular deep learning representative.

The historical LSTM result was retained as the representative temporal deep learning model.

The GRU, XGBoost, Logistic Regression, LightGBM, and Autoencoder will not be evaluated on the final test set because they did not meet the selection criteria established during validation.

The selected models are therefore:

1. Random Forest
2. MLP
3. LSTM

Random Forest remains selected because it led on validation AP. MLP and LSTM are retained as competitive representatives, and test results do not retroactively change model selection. This notebook performs the first final test evaluation of these preselected candidates. The 0.5 threshold is a standard diagnostic threshold, not a production threshold.

In [ ]:
# Import standard libraries used for model loading and evaluation.
import json
from pathlib import Path

# Import numerical and tabular data libraries.
import numpy as np
import pandas as pd

# Import joblib for loading scikit-learn model artifacts.
import joblib

# Import PyTorch for loading neural network artifacts.
import torch

# Import project evaluation metrics.
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Discover the repository root from Jupyter's current directory. This supports
# launches from either the repository root or its notebooks/ directory.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "notebooks").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the Bank-Fraud-Detection project root.")
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Set the random seed used throughout the project.
RANDOM_SEED = 42

# Select the computation device for PyTorch models.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Display the configured paths and device.
print(f"Models directory: {MODELS_DIR.resolve()}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"PyTorch device: {device}")

In [2]:
# Load the Random Forest artifact saved with joblib.
random_forest_model = joblib.load(
    MODELS_DIR / "random_forest.joblib"
)

# Load the Random Forest metadata.
with open(MODELS_DIR / "random_forest_metadata.json", "r") as file:
    random_forest_metadata = json.load(file)

# Load the MLP metadata.
with open(MODELS_DIR / "mlp_metadata.json", "r") as file:
    mlp_metadata = json.load(file)

# Load the LSTM metadata.
with open(MODELS_DIR / "lstm_metadata.json", "r") as file:
    lstm_metadata = json.load(file)

# Display the loaded Random Forest configuration.
print("Random Forest artifact loaded.")
print(f"Number of features: {len(random_forest_metadata['features'])}")

# Display the stored MLP configuration.
print("\nMLP metadata loaded.")
print(f"Number of features: {len(mlp_metadata['features'])}")

# Display the stored LSTM configuration.
print("\nLSTM metadata loaded.")
print(f"Number of features: {len(lstm_metadata['features'])}")

Random Forest artifact loaded.
Number of features: 32

MLP metadata loaded.
Number of features: 32

LSTM metadata loaded.
Number of features: 32


In [3]:
# Define the MLP architecture used during Tier 2 training.
class MLPClassifier(torch.nn.Module):
    # Match the named layers used by src/train_models.py for state_dict loading.
    def __init__(self, input_size=32):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_size, 64)
        self.relu1 = torch.nn.ReLU()
        self.dropout1 = torch.nn.Dropout(0.20)
        self.fc2 = torch.nn.Linear(64, 32)
        self.relu2 = torch.nn.ReLU()
        self.dropout2 = torch.nn.Dropout(0.20)
        self.output = torch.nn.Linear(32, 1)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        return self.output(x)


# Match the named layers used by src/train_models.py for state_dict loading.
class LSTMClassifier(torch.nn.Module):
    def __init__(self, input_size=32, hidden_size=64):
        super().__init__()
        self.lstm = torch.nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )
        self.output_layer = torch.nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (hidden_state, _) = self.lstm(x)
        return self.output_layer(hidden_state[-1]).squeeze(1)

# Create the MLP model with the original architecture.
mlp_model = MLPClassifier(input_size=32).to(device)

# Load the saved MLP weights.
mlp_model.load_state_dict(
    torch.load(
        MODELS_DIR / "mlp_state_dict.pt",
        map_location=device,
        weights_only=True,
    ),
    strict=True,
)

# Set the MLP to evaluation mode.
mlp_model.eval()


# Create the LSTM model with the original architecture.
lstm_model = LSTMClassifier(
    input_size=32,
    hidden_size=64,
).to(device)

# Load the saved LSTM weights.
lstm_model.load_state_dict(
    torch.load(
        MODELS_DIR / "lstm_state_dict.pt",
        map_location=device,
        weights_only=True,
    ),
    strict=True,
)

# Set the LSTM to evaluation mode.
lstm_model.eval()


# Confirm that all selected models are ready for evaluation.
print("Random Forest: loaded")
print("MLP: loaded")
print("LSTM: loaded")
print(f"Device for neural models: {device}")

Random Forest: loaded
MLP: loaded
LSTM: loaded
Device for neural models: cpu


## 2. Final Test-Set Integrity Checks

This notebook performs the first final model evaluation on the test set. It loads only `X_test.parquet` and `y_test.parquet` to verify dimensions, feature order, class distribution, missing values, and row alignment before scoring the preselected models. Training and validation data are not loaded in this notebook.

In [4]:
# Load only the test feature matrix and target for final-evaluation readiness checks.
X_test = pd.read_parquet(DATA_DIR / "X_test.parquet")
y_test = pd.read_parquet(DATA_DIR / "y_test.parquet").squeeze("columns")

# Compare the test feature order against each selected model's stored feature contract.
selected_feature_lists = {
    "Random Forest": random_forest_metadata["features"],
    "MLP": mlp_metadata["features"],
    "LSTM": lstm_metadata["features"],
}
expected_features = random_forest_metadata["features"]
assert len(expected_features) == 32, "Selected model metadata must contain 32 features."
assert all(features == expected_features for features in selected_feature_lists.values()), \
    "Selected model feature lists do not match one another."
assert X_test.shape[1] == 32, f"Expected 32 test features, got {X_test.shape[1]}."
assert X_test.columns.tolist() == expected_features, \
    "Test feature order does not match the selected model metadata."

# Confirm that the chronological 70/15/15 split produced its expected test size.
EXPECTED_TEST_OBSERVATIONS = 42_722
assert len(X_test) == EXPECTED_TEST_OBSERVATIONS, \
    f"Expected {EXPECTED_TEST_OBSERVATIONS} test rows, got {len(X_test)}."
assert len(y_test) == EXPECTED_TEST_OBSERVATIONS, \
    f"Expected {EXPECTED_TEST_OBSERVATIONS} test labels, got {len(y_test)}."
assert len(X_test) == len(y_test), "Test features and labels have different row counts."

# Report test-set composition and data-integrity checks without making predictions.
test_class_counts = y_test.value_counts().reindex([0, 1], fill_value=0).sort_index()
test_fraud_rate = float((y_test == 1).mean())
test_feature_missing_values = int(X_test.isna().to_numpy().sum())
test_target_missing_values = int(y_test.isna().sum())
assert test_feature_missing_values == 0, "Test features contain missing values."
assert test_target_missing_values == 0, "Test target contains missing values."

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Expected test observations: {EXPECTED_TEST_OBSERVATIONS}")
print(f"Feature count: {X_test.shape[1]}")
print(f"Feature order matches all selected model metadata: {X_test.columns.tolist() == expected_features}")
print("Test class counts (0 = legitimate, 1 = fraud):")
print(test_class_counts.to_string())
print(f"Test fraud rate: {test_fraud_rate:.6%}")
print(f"Missing values in X_test: {test_feature_missing_values}")
print(f"Missing values in y_test: {test_target_missing_values}")
print(f"Feature and target row counts match: {len(X_test) == len(y_test)}")
print("No predictions or test performance metrics have been calculated.")

X_test shape: (42722, 32)
y_test shape: (42722,)
Expected test observations: 42722
Feature count: 32
Feature order matches all selected model metadata: True
Test class counts (0 = legitimate, 1 = fraud):
Class
0    42670
1       52
Test fraud rate: 0.121717%
Missing values in X_test: 0
Missing values in y_test: 0
Feature and target row counts match: True
No predictions or test performance metrics have been calculated.


## 3. Final-Evaluation Fraud Probabilities

Generate fraud probabilities for the selected Random Forest, MLP, and LSTM using the test split only. Random Forest receives the original feature values; MLP and LSTM receive the same RobustScaler transformation used during their training. The LSTM uses chronological sliding windows of length 10, constructed wholly within the test split. Therefore, the first 9 test transactions do not become sequence targets, yielding 42,713 LSTM probabilities from 42,722 test transactions. No threshold or test metric is applied in this stage.

In [5]:
# Apply the saved training-fitted scaler to the already loaded test features for neural models.
# Loading the fitted transformer does not load or use any training or validation rows.
robust_scaler = joblib.load(DATA_DIR / "robust_scaler.joblib")
X_test_scaled_for_neural = pd.DataFrame(
    robust_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)
assert X_test_scaled_for_neural.columns.tolist() == expected_features

# Random Forest probabilities use the unscaled test features.
random_forest_fraud_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

# MLP logits are converted to fraud probabilities with sigmoid in chronological row order.
mlp_features_tensor = torch.from_numpy(
    X_test_scaled_for_neural.to_numpy(dtype=np.float32, copy=True)
).to(device)
mlp_probability_batches = []
with torch.inference_mode():
    for batch_start in range(0, len(mlp_features_tensor), 1024):
        batch = mlp_features_tensor[batch_start:batch_start + 1024]
        logits = mlp_model(batch).squeeze(1)
        mlp_probability_batches.append(torch.sigmoid(logits).cpu().numpy())
mlp_fraud_probabilities = np.concatenate(mlp_probability_batches)

# Build non-overlapping-boundary-safe rolling sequences within the test split only.
SEQUENCE_LENGTH = int(lstm_metadata["configuration"]["sequence_length"])
test_feature_values = X_test_scaled_for_neural.to_numpy(dtype=np.float32, copy=True)
X_test_sequences = np.stack([
    test_feature_values[start:start + SEQUENCE_LENGTH]
    for start in range(len(test_feature_values) - SEQUENCE_LENGTH + 1)
]).astype(np.float32, copy=False)
assert SEQUENCE_LENGTH == 10
assert X_test_sequences.shape == (len(X_test) - SEQUENCE_LENGTH + 1, SEQUENCE_LENGTH, 32)

# Run LSTM inference in ordered batches and convert logits to fraud probabilities.
lstm_features_tensor = torch.from_numpy(X_test_sequences).to(device)
lstm_probability_batches = []
with torch.inference_mode():
    for batch_start in range(0, len(lstm_features_tensor), 1024):
        batch = lstm_features_tensor[batch_start:batch_start + 1024]
        logits = lstm_model(batch)
        lstm_probability_batches.append(torch.sigmoid(logits).cpu().numpy())
lstm_fraud_probabilities = np.concatenate(lstm_probability_batches)

# Validate prediction lengths, finite values, and probability bounds without scoring.
expected_prediction_lengths = {
    "Random Forest": 42_722,
    "MLP": 42_722,
    "LSTM": 42_713,
}
fraud_probability_predictions = {
    "Random Forest": random_forest_fraud_probabilities,
    "MLP": mlp_fraud_probabilities,
    "LSTM": lstm_fraud_probabilities,
}
for model_name, probabilities in fraud_probability_predictions.items():
    assert probabilities.shape == (expected_prediction_lengths[model_name],)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0.0) & (probabilities <= 1.0)).all()

# Display only basic probability diagnostics; no classification metrics are calculated.
for model_name, probabilities in fraud_probability_predictions.items():
    print(f"{model_name}:")
    print(f"  shape: {probabilities.shape}")
    print(f"  minimum probability: {probabilities.min():.8f}")
    print(f"  maximum probability: {probabilities.max():.8f}")
    print(f"  mean probability: {probabilities.mean():.8f}")
    print(f"  finite values: {int(np.isfinite(probabilities).sum())}")

Random Forest:
  shape: (42722,)
  minimum probability: 0.00000000
  maximum probability: 0.96500000
  mean probability: 0.00138196
  finite values: 42722
MLP:
  shape: (42722,)
  minimum probability: 0.00000000
  maximum probability: 1.00000000
  mean probability: 0.01670110
  finite values: 42722
LSTM:
  shape: (42713,)
  minimum probability: 0.00036882
  maximum probability: 0.99932766
  mean probability: 0.02897878
  finite values: 42713


## 4. Final Test-Set Metrics

Evaluate only the selected Random Forest, MLP, and LSTM predictions already generated above. Average Precision and ROC-AUC use continuous fraud probabilities. Precision, Recall, F1-score, and the confusion matrix use the standard descriptive threshold of 0.5; no test-set threshold optimization or operating-threshold selection is performed.

The LSTM evaluation uses labels from the final transaction of each length-10 test sequence. The first 9 original test transactions are excluded because they cannot form a complete sequence.

In [6]:
# Align each already-generated probability array with its corresponding test labels.
y_test_full = y_test.to_numpy(dtype=np.int64, copy=True)
y_test_lstm = y_test.iloc[SEQUENCE_LENGTH - 1:].to_numpy(dtype=np.int64, copy=True)
assert len(y_test_full) == 42_722
assert len(y_test_lstm) == 42_713
assert len(y_test_full) == len(random_forest_fraud_probabilities) == len(mlp_fraud_probabilities)
assert len(y_test_lstm) == len(lstm_fraud_probabilities)

# Calculate ranking metrics from continuous probabilities and threshold metrics at 0.5.
evaluation_inputs = [
    ("Random Forest", random_forest_fraud_probabilities, y_test_full),
    ("MLP", mlp_fraud_probabilities, y_test_full),
    ("LSTM", lstm_fraud_probabilities, y_test_lstm),
]
final_test_rows = []
for model_name, probabilities, targets in evaluation_inputs:
    binary_predictions = (probabilities >= 0.5).astype(np.int64)
    matrix = confusion_matrix(targets, binary_predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    final_test_rows.append({
        "Model": model_name,
        "AP": average_precision_score(targets, probabilities),
        "Precision @ 0.5": precision_score(targets, binary_predictions, zero_division=0),
        "Recall @ 0.5": recall_score(targets, binary_predictions, zero_division=0),
        "F1 @ 0.5": f1_score(targets, binary_predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(targets, probabilities),
        "Confusion Matrix [ [TN, FP], [FN, TP] ]": f"[[{tn}, {fp}], [{fn}, {tp}]]",
    })

# Present the final test results; no threshold is selected from these results.
final_test_metrics = pd.DataFrame(final_test_rows).set_index("Model")
print(final_test_metrics.to_string(float_format=lambda value: f"{value:.6f}"))

                    AP  Precision @ 0.5  Recall @ 0.5  F1 @ 0.5  ROC-AUC Confusion Matrix [ [TN, FP], [FN, TP] ]
Model                                                                                                           
Random Forest 0.752371         0.972222      0.673077  0.795455 0.937687                  [[42669, 1], [17, 35]]
MLP           0.753003         0.167347      0.788462  0.276094 0.964874                [[42466, 204], [11, 41]]
LSTM          0.740742         0.106952      0.769231  0.187793 0.982393                [[42327, 334], [12, 40]]


## 5. Final Evaluation Interpretation

### Generalization from validation to test

All three selected models had lower Average Precision (AP) on the test set than on the validation set: Random Forest decreased from 0.858000 to 0.752371, MLP from 0.857221 to 0.753003, and LSTM from 0.848276 to 0.740742. Validation ranking performance therefore did not fully carry over to unseen transactions. This is an important generalization finding; these results alone do not establish why the reduction occurred.

### Model results

Random Forest was the validation leader with AP 0.858000. Its test AP was 0.752371. At the standard threshold of 0.5, it achieved 97.22% precision, 67.31% recall, and 79.55% F1, with one false positive and 35 of the 52 fraud cases detected. This is strong threshold-specific behavior when limiting false alerts is important. The threshold was not optimized on the test set.

MLP achieved test AP 0.753003, just 0.000632 above Random Forest. At threshold 0.5, MLP had higher recall (78.85%) but much lower precision (16.73%), with 204 false positives. This reflects a different precision-recall trade-off and does not make MLP superior overall.

LSTM achieved test AP 0.740742 and the highest ROC-AUC among these models at 0.982393; AP remains the project's primary ranking metric. At threshold 0.5, LSTM generated 334 false positives, substantially more than Random Forest. Its sequences represent windows of 10 globally chronological transactions. Because the dataset has no customer, account, merchant, or device identifiers, these windows do not represent individual entities' transaction histories. The first 9 test transactions were excluded from LSTM target evaluation because they cannot form a complete sequence.

### Final model and threshold

Random Forest remains the final model selected for this project because it achieved the highest AP during validation, before final test evaluation. The test results assess the generalization of that preselected model; they were not used to select a model after the fact. Although MLP's test AP is marginally higher, the models are essentially tied on test AP, while Random Forest has substantially higher threshold-0.5 precision and F1 and far fewer false positives.

The 0.5 threshold is only the standard diagnostic threshold used for this evaluation. The 0.5 threshold is not a production threshold. No production operating threshold has been selected or optimized using the test set. Test data was excluded from fitting, hyperparameter tuning, early stopping, model selection, and threshold optimization, although its distribution was inspected earlier during EDA and integrity analysis. The results are a held-out evaluation rather than a strictly blind holdout.